# Demo 04 — Vector Stores

The runnable notebook behind **§7 (Vector Stores)** of *Chapter 04 — Inference-Time Retrieval
Patterns*. A list of vectors in memory isn't a database; production retrieval needs a real store
that **persists**, **filters**, and **updates**. We build the same collection in **Chroma** (embedded)
and **Qdrant** (server-grade, run here in-memory), tune the ANN index by measuring recall, and close
with a DSPy retriever that supports **on-the-fly `add()`** — the capability the in-memory
`dspy.retrievers.Embeddings` lacks.

**Corpus.** `jamescalam/ai-arxiv-chunked` — chunked arXiv AI papers with `title` / `category` /
`year` metadata, so metadata filtering is meaningful.

**No LLM API key needed.** Everything runs locally: local embeddings, Chroma on disk, Qdrant
in-memory. (Qdrant Cloud is one line away, but we avoid it — no server, no credential.)

## Part 1 — Setup

macOS notes (same as `demo04_retrieval_eval`): MPS segfaults for this torch build, so we use CPU;
`torch` + `faiss` ship clashing OpenMP runtimes, so we allow the duplicate and pin faiss to one thread.

In [ ]:
# Dependencies:
# !pip install datasets sentence-transformers faiss-cpu chromadb qdrant-client python-dotenv matplotlib pandas

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"      # torch + faiss OpenMP coexistence (macOS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()
faiss.omp_set_num_threads(1)

EMBED_MODEL = "all-MiniLM-L6-v2"   # same embedder as the rest of the chapter
EMBED_DIM = 384
N_CHUNKS = 5000                    # a stable subset, big enough for real filters and ANN tuning
CACHE = Path("vector_stores_cache")
CACHE.mkdir(exist_ok=True)

model = SentenceTransformer(EMBED_MODEL, device="cpu")

def embed(texts, normalize=True):
    return np.asarray(model.encode(list(texts), batch_size=256, normalize_embeddings=normalize,
                                   convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)

def embed_query(text):
    return embed([text])[0]

## Part 2 — Load the corpus, build metadata, embed

**Keep only the metadata you will filter on or display** — never mass-copy every column; metadata
costs disk, memory, and insert time. Embeddings are cached to disk so reruns are instant.

In [ ]:
from datasets import load_dataset

ds = load_dataset("jamescalam/ai-arxiv-chunked", split="train").select(range(N_CHUNKS))
texts = [r["chunk"] for r in ds]

def make_metadata(r):
    pub = (r.get("published") or "")[:10]
    year = int(pub[:4]) if pub[:4].isdigit() else 0
    return {"title": (r.get("title") or "").strip(),
            "category": r.get("primary_category") or "",
            "year": year}

metadatas = [make_metadata(r) for r in ds]
ids = [f"chunk_{i:05d}" for i in range(N_CHUNKS)]

emb_path = CACHE / f"emb_{N_CHUNKS}_{EMBED_MODEL}.npy"
if emb_path.exists():
    embeddings = np.load(emb_path)
else:
    embeddings = embed(texts)
    np.save(emb_path, embeddings)
assert embeddings.shape == (N_CHUNKS, EMBED_DIM)
print(f"{N_CHUNKS:,} chunks embedded. Example metadata: {metadatas[0]}")

## Part 3 — Why a list — or FAISS — isn't enough (§7.1)

The toy retriever is one matrix multiply; it works at 10k vectors and dies past 1M — and it cannot
persist, filter, or update without you writing all of that yourself. **FAISS** (Facebook AI
Similarity Search) fixes speed (fast ANN) but is a *library*, not a database: still no metadata
filter, no persistence protocol, no concurrent updates. That gap is what a vector store fills.

In [ ]:
q = "What is self-attention and why does it work?"
qv = embed_query(q)

# Toy: brute-force cosine over the whole matrix (vectors are normalized -> dot = cosine)
sims = embeddings @ qv
top = np.argsort(-sims)[:5]
print("numpy brute force:")
for i in top:
    print(f"  [{sims[i]:.3f}] {metadatas[i]['title'][:60]}")

# FAISS: same result, but an actual ANN index (here Flat = exact). Still no filter/persist/update.
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)
_, faiss_top = index.search(qv[None], 5)
print("\nfaiss IndexFlatIP:", faiss_top[0].tolist())

| Need | numpy / list | FAISS | Vector DB |
|---|---|---|---|
| Top-k search | O(N·d) scan | brute or ANN | ANN |
| Persistent storage | manual pickle | manual | ✅ |
| Add / delete after build | manual | rebuild | ✅ |
| Metadata filtering | ✗ | ✗ | ✅ |
| Concurrency, multi-tenant | ✗ | ✗ | ✅ |

## Part 4 — Chroma: the prototyping default (§7.3)

Chroma runs in-process, `pip`-installs, and persists to a folder. Three calls cover most use:
`add`, `query`, `update`/`delete`. Under the hood it is HNSW (Hierarchical Navigable Small World) +
SQLite for metadata.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(CACHE / "chroma_db"))
col = client.get_or_create_collection("arxiv", metadata={"hnsw:space": "cosine"})

if col.count() == 0:
    BATCH = 500
    for i in range(0, N_CHUNKS, BATCH):
        j = min(i + BATCH, N_CHUNKS)
        col.add(ids=ids[i:j], documents=texts[i:j],
                embeddings=embeddings[i:j].tolist(), metadatas=metadatas[i:j])
print("Documents in collection:", col.count())

In [ ]:
# Plain similarity search
res = col.query(query_embeddings=[qv.tolist()], n_results=5)
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"[{dist:.3f}] {meta['title'][:55]}  ({meta['category']}, {meta['year']})")

In [ ]:
# Similarity + structured filter (Chroma uses Mongo-style operators)
res = col.query(query_embeddings=[qv.tolist()], n_results=5,
                where={"$and": [{"category": {"$eq": "cs.CL"}}, {"year": {"$gte": 2020}}]})
print("category=cs.CL AND year>=2020:")
for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
    print(f"  [{dist:.3f}] {meta['title'][:55]}  ({meta['category']}, {meta['year']})")

In [ ]:
# Update / delete by id — the thing a plain list can't do without a rebuild
col.update(ids=["chunk_00000"], metadatas=[{**metadatas[0], "reviewed": True}])
print("After update:", col.get(ids=["chunk_00000"])["metadatas"][0])

## Part 5 — Qdrant: server-grade, run in-memory (§7.4)

Qdrant is a production store (Rust) with rich payload filters, hybrid search, and quantization. The
shape is the same — create, upsert, query — with explicit vector config and first-class filters.
`QdrantClient(":memory:")` runs the real engine locally with no server and no credential; switching
to a URL + API key is the only change for Qdrant Cloud.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct,
    Filter, FieldCondition, Range, MatchValue,
    HnswConfigDiff, SearchParams,
)

qdrant = QdrantClient(":memory:")   # real engine, local; no server, no secret
COLL = "arxiv"

qdrant.create_collection(
    collection_name=COLL,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=16, ef_construct=64),   # defaults made explicit
)

BATCH = 500
for i in range(0, N_CHUNKS, BATCH):
    j = min(i + BATCH, N_CHUNKS)
    qdrant.upsert(collection_name=COLL, points=[
        PointStruct(id=k, vector=embeddings[k].tolist(),
                    payload={**metadatas[k], "text": texts[k]})
        for k in range(i, j)
    ])

# Index every payload field you filter on (required by Qdrant Cloud's strict mode).
qdrant.create_payload_index(collection_name=COLL, field_name="category", field_schema="keyword")
qdrant.create_payload_index(collection_name=COLL, field_name="year", field_schema="integer")
print("Points in collection:", qdrant.get_collection(COLL).points_count)

In [ ]:
# Plain search, then the same query with a payload filter
hits = qdrant.query_points(collection_name=COLL, query=qv.tolist(), limit=5, with_payload=True).points
print("Top hits:")
for h in hits:
    print(f"  [{h.score:.3f}] {h.payload['title'][:55]}  ({h.payload['category']}, {h.payload['year']})")

hits = qdrant.query_points(
    collection_name=COLL, query=qv.tolist(), limit=5, with_payload=True,
    query_filter=Filter(must=[
        FieldCondition(key="category", match=MatchValue(value="cs.CL")),
        FieldCondition(key="year", range=Range(gte=2020)),
    ]),
).points
print("\ncategory=cs.CL AND year>=2020:")
for h in hits:
    print(f"  [{h.score:.3f}] {h.payload['title'][:55]}  ({h.payload['category']}, {h.payload['year']})")

## Part 6 — Tuning the ANN index: measure recall first (§7.5–§7.6)

You cannot tune what you don't measure. Build a small ground-truth set with exact brute-force top-k,
then sweep the one knob you touch in production — HNSW `ef_search` — and read recall against latency.
The elbow, where recall plateaus, is the setting to ship.

In [ ]:
K = 10
rng = np.random.default_rng(42)
eval_idx = rng.choice(N_CHUNKS, size=50, replace=False)
eval_vecs = embeddings[eval_idx]

# Exact reference (normalized vectors -> cosine = dot)
sims = eval_vecs @ embeddings.T
exact_sets = [set(np.argsort(-sims[i])[:K].tolist()) for i in range(len(eval_idx))]

def measure(ef):
    recalls, lats = [], []
    for vec, truth in zip(eval_vecs, exact_sets):
        t0 = time.perf_counter()
        hits = qdrant.query_points(collection_name=COLL, query=vec.tolist(), limit=K,
                                   search_params=SearchParams(hnsw_ef=ef), with_payload=False).points
        lats.append((time.perf_counter() - t0) * 1000)
        recalls.append(len({h.id for h in hits} & truth) / K)
    return float(np.mean(recalls)), float(np.median(lats))

ef_values = [10, 20, 40, 80, 160, 320]
rows = [(ef, *measure(ef)) for ef in ef_values]
df_ef = pd.DataFrame(rows, columns=["ef_search", "recall@10", "median_ms"])
print(df_ef.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_ef["median_ms"], df_ef["recall@10"], marker="o")
for _, r in df_ef.iterrows():
    ax.annotate(f"ef={int(r['ef_search'])}", (r["median_ms"], r["recall@10"]),
                textcoords="offset points", xytext=(5, -8), fontsize=8)
ax.set_xlabel("median query latency (ms)"); ax.set_ylabel("recall@10 vs exact")
ax.set_title("HNSW ef_search: recall vs latency (Qdrant)"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Part 7 — Metadata and pre/post-filtering (§7.7)

Vectors are the *where* (semantic); metadata is the *what* (categorical). A selective filter is the
case that separates a real store from a list: Qdrant pre-filters using the payload index, so a query
restricted to a rare category stays fast instead of running out of candidates.

In [ ]:
from collections import Counter

cat_counts = Counter(m["category"] for m in metadatas)
common = cat_counts.most_common(1)[0][0]
rare = cat_counts.most_common()[-1][0]

def time_filtered(category, n=20):
    flt = Filter(must=[FieldCondition(key="category", match=MatchValue(value=category))])
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        qdrant.query_points(collection_name=COLL, query=qv.tolist(), limit=10,
                            query_filter=flt, with_payload=False)
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))

print(f"common ({common:<10s} {cat_counts[common]:>4d} chunks): {time_filtered(common):.2f} ms")
print(f"rare   ({rare:<10s} {cat_counts[rare]:>4d} chunks): {time_filtered(rare):.2f} ms")

## Part 8 — A DSPy retriever with on-the-fly `add()`

The §3 build used `dspy.retrievers.Embeddings` — vectors in RAM, **immutable**: adding a document
means re-embedding the whole corpus. Backing a small `dspy.Module` with a store fixes that. The store
owns a mutable, persistent index, so `.add()` makes a new document retrievable **immediately** — no
rebuild. This is the idiomatic way to point DSPy at any vector database (§7.2).

In [ ]:
import dspy

class ChromaRetriever(dspy.Module):
    """A DSPy retriever backed by a Chroma collection — supports live add()."""
    def __init__(self, collection, embed_fn, k=5):
        self.collection, self.embed_fn, self.k = collection, embed_fn, k

    def add(self, docs, ids, metadatas=None):
        self.collection.add(ids=ids, documents=docs,
                            embeddings=[self.embed_fn(d).tolist() for d in docs],
                            metadatas=metadatas)

    def forward(self, query):
        res = self.collection.query(query_embeddings=[self.embed_fn(query).tolist()],
                                    n_results=self.k)
        return dspy.Prediction(passages=res["documents"][0])

retriever = ChromaRetriever(col, embed_query, k=3)

# A fact no arXiv chunk contains — retrieval misses it...
probe = "What is the internal codename of Project Zephyr?"
print("Before add:", retriever(probe).passages[0][:80], "...")

# ...add it live, and it is retrievable on the very next query — no re-index.
retriever.add(docs=["Project Zephyr is the internal codename for the 2026 retrieval benchmark."],
              ids=["live_001"], metadatas=[{"title": "Zephyr note", "category": "internal", "year": 2026}])
print("After add: ", retriever(probe).passages[0])

## Summary

- A **list/FAISS** gives you search but not persistence, filtering, or live updates — a store does
  all three (§7.1).
- **Chroma** (embedded, HNSW + SQLite) is the prototyping default; **Qdrant** (server-grade, here
  in-memory) adds rich payload filters, hybrid, and quantization (§7.3–§7.4).
- **Tune by measuring recall** against exact top-k, then pick the `ef_search` elbow (§7.5–§7.6).
- **Pre-filtering** with a payload index keeps selective metadata queries fast (§7.7).
- Backing a **`dspy.Module`** with a store gives DSPy retrieval **live `add()`** — impossible with
  the in-memory `Embeddings` retriever.

Hybrid (dense + sparse) search and reranking build on this store — that is §8 (`demo04_rerankers`).